In [1]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Route between specialist agents

| | |
|-|-|
| Author(s) | [Matt Robinson](https://github.com/mr394729) |

> **This copy keeps the output of one complete run** (24 September 2026, in a test namespace), so you can read what each cell prints even if a cell fails for you. Your numbers and wording will differ where a model answers. To start clean, choose **Edit > Clear Outputs of All Cells** in JupyterLab, or run `jupyter nbconvert --clear-output --inplace <notebook>`.

## Overview

### Sub-agents and their descriptions

An ADK agent can have [sub-agents](https://adk.dev/workflows/collaboration/). The coordinator does not need routing rules: it reads each sub-agent's `description` and decides which one fits the question. Good descriptions are the routing table.

### Two ways a sub-agent joins in

- A **chat** sub-agent takes over the conversation. The coordinator transfers control with `transfer_to_agent`, and the sub-agent talks to the user from then on.
- A **single-turn** sub-agent (`mode="single_turn"`) is called like a tool. It receives only the inputs in its `input_schema`, answers once, and the coordinator keeps control.

### Workflow patterns

When the order of steps is always the same, you write it in code with workflow agents instead of letting a model choose: [`SequentialAgent`](https://adk.dev/agents/workflow-agents/sequential-agents/), [`ParallelAgent`](https://adk.dev/agents/workflow-agents/parallel-agents/) and [`LoopAgent`](https://adk.dev/agents/workflow-agents/loop-agents/). The `patterns/` folder has one short script for each, all on the store data.

<img width="60%" src="../../docs/diagrams/q10.png" alt="A coordinator transfers to a chat specialist or calls a single-turn specialist as a tool" />

### Objectives

In this tutorial, you will learn how an ADK coordinator hands work to other agents, and when to use a workflow agent instead.

You will complete the following tasks:

- Read the router's sub-agents, their modes and descriptions
- Ask a stock question and see a single-turn call
- Ask a coaching question and see a transfer
- Build and run a generate-and-review loop

### Costs

This tutorial uses billable components of Google Cloud:

- Gemini on Vertex AI
- BigQuery

Learn about [Vertex AI pricing](https://cloud.google.com/vertex-ai/pricing) and [BigQuery pricing](https://cloud.google.com/bigquery/pricing), and use the [Pricing Calculator](https://cloud.google.com/products/calculator/) to generate a cost estimate based on your projected usage.

## Get started

### Set Google Cloud project information

This quickstart reads the store data you loaded in the workshop notebooks, in your own namespace. Set your project ID and the namespace you chose during setup.

Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [2]:
import os
import sys
from pathlib import Path

PROJECT_ID = "[your-project-id]"  # @param {type: "string"}
WORKSHOP_NAMESPACE = "[your-namespace]"  # @param {type: "string"}

if PROJECT_ID == "[your-project-id]":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "")
if WORKSHOP_NAMESPACE == "[your-namespace]":
    WORKSHOP_NAMESPACE = os.environ.get("WORKSHOP_NAMESPACE", "")
if not PROJECT_ID or not WORKSHOP_NAMESPACE:
    raise ValueError("Set PROJECT_ID and WORKSHOP_NAMESPACE above.")

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["WORKSHOP_NAMESPACE"] = WORKSHOP_NAMESPACE
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["STORE_OPS_ENV"] = "dev"

# The quickstart imports shared store code from the repository root, two folders up
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "agents" / "cymbal_store_ops").is_dir())
sys.path.insert(0, str(REPO_ROOT))

### Import libraries

In [3]:
import logging
import warnings

# Keep the output to the agent's own events: ADK marks experimental and deprecated features
# with warnings and logs configuration hints, and the Gen AI SDK logs a note whenever a
# response mixes text and tool calls.
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("google_adk").setLevel(logging.ERROR)
logging.getLogger("google_genai").setLevel(logging.ERROR)

In [4]:
from agent import app
from google.adk.agents import LlmAgent, LoopAgent
from google.adk.runners import InMemoryRunner
from google.adk.tools import exit_loop
from google.genai import types

from agents.cymbal_store_ops.tools.domain_tools import get_shift_roster, get_traffic_and_backlog

## Look at the router

`agent.py` in this folder defines a coordinator with two sub-agents. The coordinator's instruction names no tools; it relies on each sub-agent's description:

In [5]:
router = app.root_agent

for sub_agent in router.sub_agents:
    print(f"{sub_agent.name}  mode={sub_agent.mode}")
    print(f"  {sub_agent.description}")
    if sub_agent.input_schema:
        print(f"  input: {list(sub_agent.input_schema.model_fields)}")

coaching_specialist  mode=chat
  Coaching and development questions about an associate: signals, training suggestions.
stock_lookup  mode=single_turn
  How much of one product do the stores in one city hold, on shelf and in the backroom?
  input: ['product_name', 'city']


## Run the router

Start a session and define a helper that prints each tool call, each transfer and the answer.

In [6]:
runner = InMemoryRunner(app=app)
session = await runner.session_service.create_session(app_name=app.name, user_id="dana")

In [7]:
async def ask(question: str) -> None:
    """Send one message and print the tool calls, transfers and final answer."""
    message = types.Content(role="user", parts=[types.Part(text=question)])
    async for event in runner.run_async(
        user_id=session.user_id, session_id=session.id, new_message=message
    ):
        for call in event.get_function_calls():
            if call.name == "transfer_to_agent":
                print(f"[{event.author}] hands over to {call.args.get('agent_name')}")
            else:
                print(f"[{event.author}] calls {call.name}({dict(call.args or {})})")
        if event.is_final_response() and event.content and event.content.parts:
            text = "".join(part.text or "" for part in event.content.parts if not part.thought)
            if text:
                print(f"\n[{event.author}] {text}\n")

### A single-turn call

A stock question fits `stock_lookup`. The coordinator calls it like a tool, with a `product_name` and a `city`, and writes the answer itself:

In [8]:
await ask("How much Lumière Hydra Cream do the Naperville stores hold?")

[multi_agent_router] calls stock_lookup({'product_name': 'Lumière Hydra Cream', 'city': 'Naperville'})


[stock_lookup] calls check_store_stock({'city': 'Naperville', 'product_name': 'Lumière Hydra Cream'})



[stock_lookup] At Cymbal Beauty Naperville, there are 0 units on-shelf, 7 units in the backroom, and a total of 7 units on-hand.




[multi_agent_router] At Cymbal Beauty Naperville, there are 0 units of Lumière Hydra Cream on-shelf, 7 units in the backroom, for a total of 7 units on-hand.



The `stock_lookup` agent receives only `product_name` and `city`, calls `check_store_stock`, and returns 0 on the shelf, 7 in the backroom and 7 on hand. The coordinator then writes its own answer from that result.

### A transfer

A coaching question fits `coaching_specialist`. The coordinator transfers the conversation, and the specialist signs Dana in and answers:

In [9]:
await ask("I'm U-M014. How is A-1007 doing on BOPIS picking?")

[multi_agent_router] hands over to coaching_specialist


[coaching_specialist] calls identify_demo_user({'user_id': 'U-M014'})


[coaching_specialist] calls get_coaching_signals({'associate_id': 'A-1007'})



[coaching_specialist] Noor's BOPIS pick rate is currently at 52% for week 39, indicating an opportunity to improve fulfillment speed and order picking efficiency. Other performance areas remain solid, with task completion at 94% and cycle count accuracy at 89%.

To support Noor's development, consider pairing them with a seasoned fulfillment associate for shadowing during peak order windows. Reviewing efficient pick paths and staging routines, alongside completing the digital fulfillment refresher module, could help increase picking speed.



The coordinator made no tool calls itself: it transferred, and the specialist signed Dana in with `identify_demo_user` before it read the coaching signals. The pick rate is 52 % for week 2026-W39.

Any follow-up now goes straight to the coaching specialist, because it holds the conversation:

In [10]:
await ask("What training would you suggest for her?")


[coaching_specialist] * Complete the Omnichannel Fulfillment refresher course, focusing on batch picking and route navigation.
* Review handheld inventory tools training to speed up product lookups and backroom navigation.
* Schedule peer shadowing sessions with a high-performing picker to observe real-time pacing and cart staging techniques.



## Build a generate-and-review loop

A `LoopAgent` runs its sub-agents in order, again and again, until one of them calls `exit_loop` or `max_iterations` is reached. ADK 2.x marks `LoopAgent` as deprecated in favour of a `Workflow` graph, which can express the same loop (`patterns/05_workflow_graph.py` shows a `Workflow`); `LoopAgent` still works and is the shorter way to show the idea. Here a writer drafts the shift-huddle note and a critic checks it against four rules. The writer never sees the rules, so the critic is the only thing enforcing them.

### Define the writer and the critic

The writer saves its draft in the session state with `output_key`, and the critic reads it with `{huddle_draft}` in its instruction.

In [11]:
writer = LlmAgent(
    name="huddle_writer",
    model="gemini-3.8-flash",
    tools=[get_traffic_and_backlog, get_shift_roster],
    output_key="huddle_draft",
    instruction="""Write the note {user:first_name?} reads to the team at the 09:00 shift huddle:
the pending BOPIS orders, the traffic peak and who covers BOPIS picking. On the first pass call
get_traffic_and_backlog for 4 hours and get_shift_roster with focus bopis. If a review is in the
conversation, rewrite the note fixing exactly what the latest review names. Reply with the note only.""",
)

critic = LlmAgent(
    name="huddle_critic",
    model="gemini-3.8-flash",
    tools=[exit_loop],
    instruction="""Review this shift-huddle note:
{huddle_draft}

Rules: (1) it names at least one count from the tools; (2) every associate id in it (A-####)
appears in a tool result earlier in the conversation; (3) it is under 60 words; (4) it has no HR
or disciplinary language. If all four hold, call exit_loop. Otherwise reply with one line per
failed rule: the number, what is wrong and the fix.""",
)

huddle = LoopAgent(name="huddle_note", sub_agents=[writer, critic], max_iterations=3)

### Run the loop

The tools read the signed-in store from the session, so start the session as Dana. The cell prints each agent's turn, so you can see how many rounds the loop needed.

In [12]:
loop_runner = InMemoryRunner(agent=huddle, app_name="huddle_note")
loop_session = await loop_runner.session_service.create_session(
    app_name="huddle_note",
    user_id="dana",
    state={
        "user:user_id": "U-M014",
        "user:store_id": "S-014",
        "user:role": "store_manager",
        "user:first_name": "Dana",
    },
)

message = types.Content(role="user", parts=[types.Part(text="Draft my 09:00 huddle note.")])
async for event in loop_runner.run_async(
    user_id="dana", session_id=loop_session.id, new_message=message
):
    for call in event.get_function_calls():
        print(f"[{event.author}] calls {call.name}")
    if event.content and event.content.parts and not event.get_function_calls():
        text = "".join(part.text or "" for part in event.content.parts if not part.thought)
        if text:
            print(f"[{event.author}] {text}\n")

[huddle_writer] calls get_traffic_and_backlog
[huddle_writer] calls get_shift_roster


[huddle_writer] Good morning team! We have 9 pending BOPIS orders to fulfill this morning. Traffic is expected to peak at 11:00 AM with 88 visitors, and Priya will be covering BOPIS picking. Let's have a great shift!



[huddle_critic] calls exit_loop


Each tool is called once and the critic calls `exit_loop` on the first draft, so this run needed one round. The draft has a count from each tool (9 pending BOPIS orders, a peak of 88 visitors at 11:00 AM). If the critic had named a failed rule, you would see a second `huddle_writer` turn.

The final draft is in the session state:

In [13]:
loop_session = await loop_runner.session_service.get_session(
    app_name="huddle_note", user_id="dana", session_id=loop_session.id
)
print(loop_session.state["huddle_draft"])

Good morning team! We have 9 pending BOPIS orders to fulfill this morning. Traffic is expected to peak at 11:00 AM with 88 visitors, and Priya will be covering BOPIS picking. Let's have a great shift!


### Run the other patterns

The `patterns/` folder has a script for each pattern. Each one runs a single prompt as Dana and prints every transfer, tool call and state change:

| Script | Pattern |
|---|---|
| `01_coordinator_vs_single_turn.py` | A chat transfer and a single-turn call |
| `02_sequential_osa_to_task.py` | `SequentialAgent` passing a finding to a task drafter through `output_key` |
| `03_parallel_fanout_gather.py` | `ParallelAgent` reading three signals at once, then one writer |
| `04_loop_generate_review.py` | The loop you just built |
| `05_workflow_graph.py` | A `Workflow` graph that routes an event to one of three agents |

Run one from a terminal in this folder:

```bash
uv run python patterns/03_parallel_fanout_gather.py
```

## Cleaning up

This notebook creates no cloud resources. The sessions lived in memory and end when you restart the kernel.

## What's next

- [Multi-agent systems in ADK](https://adk.dev/workflows/collaboration/)
- [Workflow agents](https://adk.dev/workflows/patterns/)
- [Quickstart guide](README.md) for running this agent in the ADK developer UI
- Next quickstart: [Ambient event agent](../11-ambient-event-agent/walkthrough.ipynb)